# RealEstatePRO

In [2]:
#import all necessary libraries
import pandas as pd
import os
from getpass import getpass
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import tqdm

In [3]:
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY") or \
    getpass("Enter LangSmith API Key: ")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"] = "aurelioai-langchain-course-agent-executor-openai"

In [4]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") \
    or getpass("Enter your OpenAI API key: ")

## DataFrame import

In [5]:
df = pd.read_csv('NY-House-Dataset.csv')

In [6]:
df.head(10)

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856
5,Brokered by Sowae Corp,House for sale,690000,5,2.000000,4004.000000,584 Park Pl,"Brooklyn, NY 11238","584 Park PlBrooklyn, NY 11238",United States,New York,Kings County,Brooklyn,Park Place,"584 Park Pl, Brooklyn, NY 11238, USA",40.674363,-73.958725
6,Brokered by Douglas Elliman - 575 Madison Ave,Condo for sale,899500,2,2.000000,2184.207862,157 W 126th St Unit 1B,"New York, NY 10027","157 W 126th St Unit 1BNew York, NY 10027",New York,New York County,New York,Manhattan,157,"157 W 126th St #1b, New York, NY 10027, USA",40.809448,-73.946777
7,Brokered by Connie Profaci Realty,House for sale,16800000,8,16.000000,33000.000000,177 Benedict Rd,"Staten Island, NY 10304","177 Benedict RdStaten Island, NY 10304",United States,New York,Richmond County,Staten Island,Benedict Road,"177 Benedict Rd, Staten Island, NY 10304, USA",40.595002,-74.106424
8,Brokered by Pantiga Group Inc.,Co-op for sale,265000,1,1.000000,750.000000,875 Morrison Ave Apt 3M,"Bronx, NY 10473","875 Morrison Ave Apt 3MBronx, NY 10473",Bronx County,The Bronx,East Bronx,Morrison Avenue,Parking lot,"Parking lot, 875 Morrison Ave #3m, Bronx, NY 1...",40.821586,-73.874089
9,Brokered by CENTURY 21 MK Realty,Co-op for sale,440000,2,1.000000,978.000000,1350 Ocean Pkwy Apt 5G,"Brooklyn, NY 11230","1350 Ocean Pkwy Apt 5GBrooklyn, NY 11230",New York,Kings County,Brooklyn,Midwood,1350,"1350 Ocean Pkwy #5g, Brooklyn, NY 11230, USA",40.615738,-73.969694


In [7]:
df.describe()

,PRICE,BEDS,BATH,PROPERTYSQFT,LATITUDE,LONGITUDE
count,4.801000e+03,4801.000000,4801.000000,4801.000000,4801.000000,4801.000000
mean,2.356940e+06,3.356801,2.373861,2184.207862,40.714227,-73.941601
std,3.135525e+07,2.602315,1.946962,2377.140894,0.087676,0.101082
min,2.494000e+03,1.000000,0.000000,230.000000,40.499546,-74.253033
25%,4.990000e+05,2.000000,1.000000,1200.000000,40.639375,-73.987143
50%,8.250000e+05,3.000000,2.000000,2184.207862,40.726749,-73.949189
75%,1.495000e+06,4.000000,3.000000,2184.207862,40.771923,-73.870638
max,2.147484e+09,50.000000,50.000000,65535.000000,40.912729,-73.702450


In [8]:
df.shape

(4801, 17)

In [9]:
#make all columns lowercase
df.columns = [col.lower() for col in df.columns]

In [10]:
df.columns

Index(['brokertitle', 'type', 'price', 'beds', 'bath', 'propertysqft',
       'address', 'state', 'main_address', 'administrative_area_level_2',
       'locality', 'sublocality', 'street_name', 'long_name',
       'formatted_address', 'latitude', 'longitude'],
      dtype='object')

### Save embeddings in FAISS

What is FAISS: TODO

In [12]:
index_path = "faiss_index_dir"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [14]:
if os.path.exists(index_path):
    # Load existing FAISS index
    vector_store = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing FAISS index.")

Loaded existing FAISS index.


In [181]:
# Create a concise textual representation for embedding
df['text_to_embed'] = (
    df['brokertitle'].astype(str) + ", " +
    df['type'].astype(str) + ", Price: " + df['price'].astype(str) + "$, " +
    "Beds: " + df['beds'].astype(str) + ", Baths: " + df['bath'].astype(str) + ", " +
    "Size: " + df['propertysqft'].astype(str) + " sqft, " +
    "Address: " + df['address'].astype(str) + ", " +
    "Locality: " + df['locality'].astype(str) + ", " +
    "State: " + df['state'].astype(str) + ", " +
    "Latitude: " + df['latitude'].astype(str) + ", Longitude: " + df['longitude'].astype(str)
)

In [182]:
df

,brokertitle,type,price,beds,bath,propertysqft,address,state,main_address,administrative_area_level_2,locality,sublocality,street_name,long_name,formatted_address,latitude,longitude,text_to_embed
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483,"Brokered by Douglas Elliman -111 Fifth Ave, C..."
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991,"Brokered by Serhant, Condo for sale, Price: 19..."
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109,"Brokered by Sowae Corp, House for sale, Price:..."
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613,"Brokered by COMPASS, Condo for sale, Price: 69..."
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856,Brokered by Sotheby's International Realty - E...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,Brokered by COMPASS,Co-op for sale,599000,1,1.000000,2184.207862,222 E 80th St Apt 3A,"Manhattan, NY 10075","222 E 80th St Apt 3AManhattan, NY 10075",New York,New York County,New York,Manhattan,222,"222 E 80th St #3a, New York, NY 10075, USA",40.774350,-73.955879,"Brokered by COMPASS, Co-op for sale, Price: 59..."
4797,Brokered by Mjr Real Estate Llc,Co-op for sale,245000,1,1.000000,2184.207862,97-40 62 Dr Unit Lg,"Rego Park, NY 11374","97-40 62 Dr Unit LgRego Park, NY 11374",United States,New York,Queens County,Queens,62nd Drive,"97-40 62nd Dr, Rego Park, NY 11374, USA",40.732538,-73.860152,"Brokered by Mjr Real Estate Llc, Co-op for sal..."
4798,Brokered by Douglas Elliman - 575 Madison Ave,Co-op for sale,1275000,1,1.000000,2184.207862,427 W 21st St Unit Garden,"New York, NY 10011","427 W 21st St Unit GardenNew York, NY 10011",United States,New York,New York County,New York,West 21st Street,"427 W 21st St, New York, NY 10011, USA",40.745882,-74.003398,"Brokered by Douglas Elliman - 575 Madison Ave,..."
4799,Brokered by E Realty International Corp,Condo for sale,598125,2,1.000000,655.000000,91-23 Corona Ave Unit 4G,"Elmhurst, NY 11373","91-23 Corona Ave Unit 4GElmhurst, NY 11373",New York,Queens County,Queens,Flushing,91-23,"91-23 Corona Ave. #4b, Flushing, NY 11373, USA",40.742770,-73.872752,"Brokered by E Realty International Corp, Condo..."


In [183]:
# Create LangChain Documents, saving only essential metadata
documents = []
for idx, row in df.iterrows():
    metadata = {
        "beds": row['beds'],
        "bath": row['bath'],
        "address": row['address'],
        "state": row['state'],
        "propertysqft": row['propertysqft'],
        "price": row['price'],
        "latitude": row['latitude'],
        "longitude": row['longitude']
    }
    doc = Document(page_content=row['text_to_embed'], metadata=metadata)
    documents.append(doc)

In [184]:
documents[0]  # Display the first document to verify

Document(metadata={'beds': 2, 'bath': 2.0, 'address': '2 E 55th St Unit 803', 'state': 'New York, NY 10022', 'propertysqft': 1400.0, 'price': 315000, 'latitude': 40.761255, 'longitude': -73.9744834}, page_content='Brokered by Douglas Elliman  -111 Fifth Ave, Condo for sale, Price: 315000$, Beds: 2, Baths: 2.0, Size: 1400.0 sqft, Address: 2 E 55th St Unit 803, Locality: New York, State: New York, NY 10022, Latitude: 40.761255, Longitude: -73.9744834')

In [185]:
embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [186]:
# Add documents to the vector store 
vector_store.add_documents(documents)


['4f8a1348-0a51-461c-938d-cdd294c93ba8',
 '97c98ea5-9fe0-42b5-a40f-06d174dd0c77',
 '20a92d63-ee3e-4c0f-a943-eccd0ab62fea',
 '1d4fc045-3927-4134-b728-2e5923ac9389',
 '5898e398-80bf-48fc-9bcd-0933a732a903',
 '5a5aceaa-c0e9-45f1-9daa-ebbf365881ec',
 'de9eb690-c719-4abc-9624-d9c8f38765b7',
 '80ca480f-4cc5-46cc-8f33-13283c2d2312',
 '2c4f9783-a66c-4d7b-965c-34f2143d6a47',
 '5f99f009-f192-444c-8207-2c2c63f1e4ba',
 '6bcac79a-8994-4470-b0c2-d37a2d6009b7',
 '8d46b104-f44e-4197-8869-171055c82d48',
 '920b637e-da17-440d-8057-ae9117a56db3',
 'afdfb219-3b41-4be4-a418-f4d4b892adb2',
 'd073664a-463c-4bbb-a20c-9e25f0621688',
 'fea6a6b2-99da-4f74-9307-e2dba6a67fcb',
 '71945696-b666-44c4-9298-acf5083e38dd',
 '6eee917e-e1ae-4375-a92f-c816cb1c4b71',
 '1de8ff6c-a2cd-4f86-b55c-0a459a205ddb',
 'ea17304a-ff9a-4287-8bce-43b3a2127a46',
 '867a9f69-69f0-4695-8775-09c072e24acc',
 '8be9972a-4934-4911-9bda-6c301d0b7cec',
 '29cdcc7a-cc75-48d6-b7cc-ad1b8f9b1f3e',
 '9d391a41-7c99-40ea-88bb-0034a64a9ff6',
 '843f1b9f-2520-

In [187]:
results = vector_store.similarity_search(
    "House in New York with 3 bedrooms and not more than 2 bathrooms between 1000000$ and 3000000$",
    k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Brokered by COMPASS, House for sale, Price: 14000000$, Beds: 3, Baths: 2.3738608579684373, Size: 23027.0 sqft, Address: 39 Eldridge St, Locality: New York, State: Manhattan, NY 10002, Latitude: 40.7157769, Longitude: -73.99357 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '39 Eldridge St', 'state': 'Manhattan, NY 10002', 'propertysqft': 23027.0, 'price': 14000000, 'latitude': 40.7157769, 'longitude': -73.99357}]
* Brokered by COMPASS, Condo for sale, Price: 3000000$, Beds: 3, Baths: 3.0, Size: 2100.0 sqft, Address: 418 E 59th St # B, Locality: New York, State: Manhattan, NY 10022, Latitude: 40.7590132, Longitude: -73.9612618 [{'beds': 3, 'bath': 3.0, 'address': '418 E 59th St # B', 'state': 'Manhattan, NY 10022', 'propertysqft': 2100.0, 'price': 3000000, 'latitude': 40.7590132, 'longitude': -73.9612618}]


In [188]:
vector_store.save_local("faiss_index_dir")

## LangChain

### Tools

In [189]:
from langchain_core.tools import tool

@tool
def retrieve(query: str, k: int) -> str:
    """Retrieve relevant real estate listings based on the query."""
    results = vector_store.similarity_search(query, k=k)
    if not results:
        return "No relevant listings found."
    
    response = "Here are some relevant listings:\n"
    for i, res in enumerate(results, 1):
        response += (f"{i}. {res.page_content} | "
                     f"Beds: {res.metadata.get('beds')}, "
                     f"Baths: {res.metadata.get('bath')}, "
                     f"Price: {res.metadata.get('price')}$, "
                     f"Address: {res.metadata.get('address')}, "
                     f"State: {res.metadata.get('state')}, "
                     f"Size: {res.metadata.get('propertysqft')} sqft, "
                     f"Latitude: {res.metadata.get('latitude')}, Longitude: {res.metadata.get('longitude')}\n")
    return response

@tool
def average_price(location: str) -> str:
    """Calculate the average price of houses in a given location."""
    results = vector_store.similarity_search(f"houses in {location}", k=20)
    if not results:
        return f"No listings found for location: {location}"
    
    prices = [res.metadata.get('price') for res in results if res.metadata.get('price') is not None]
    if not prices:
        return f"No price data available for listings in {location}"
    
    avg_price = sum(prices) / len(prices)
    return f"The average price of houses in {location} is approximately ${avg_price:,.2f}."


@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`.
    """
    return {"answer": answer, "tools_used": tools_used}





In [162]:
tools = [retrieve, average_price, final_answer]

# create tool name to function mapping
name2tool = {tool.name: tool.func for tool in tools}

In [163]:
retrieve

StructuredTool(name='retrieve', description='Retrieve relevant real estate listings based on the query.', args_schema=<class 'langchain_core.utils.pydantic.retrieve'>, func=<function retrieve at 0x000001F3FA4B1120>)

In [164]:
print(f"{retrieve.name=}\n{retrieve.description=}")

retrieve.name='retrieve'
retrieve.description='Retrieve relevant real estate listings based on the query.'


In [165]:
retrieve.args_schema.model_json_schema()

{'description': 'Retrieve relevant real estate listings based on the query.',
 'properties': {'query': {'title': 'Query', 'type': 'string'},
  'k': {'title': 'K', 'type': 'integer'}},
 'required': ['query', 'k'],
 'title': 'retrieve',
 'type': 'object'}

### Creating Agent

In [175]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant that will provide information about real estate listings and assist users in finding the right property suggesting listings based on their preferences and requirements."  

"Use tools only if the question requires real estate data such as listings, prices, or property details.  "

"For general knowledge questions or questions unrelated to real estate (for example, about parks, cities, or travel), answer directly without using any tools.  "

"When answering, after using the necessary tools, ALWAYS end with the final_answer tool.  "

"After using a tool, the output will be provided in the 'scratchpad' below. If you have an answer in the scratchpad, you should not use any more tools and instead answer directly to the user."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [176]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.0,
)

In [177]:
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.runnables.base import RunnableSerializable
from langchain_core.messages import ToolMessage
import json

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            tool_call = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            if isinstance(tool_out, str):
                # model returned a string instead of dict
                final_answer = tool_out
                tools_used = [tool_name]
            elif isinstance(tool_out, dict) and "answer" in tool_out:
                final_answer = tool_out["answer"]
                tools_used = tool_out.get("tools_used", [])
            else:
                raise ValueError(f"Unexpected tool output type: {type(tool_out)} -> {tool_out}")

            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            # add a print so we can see intermediate steps
            print(f"{count}: {tool_name}({tool_args})")
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return json.dumps(tool_out)

In [178]:
agent_executor = CustomAgentExecutor()

In [180]:
agent_executor.invoke(input="Which are the best houses near parks that I can find in Manhattan?")

0: retrieve({'query': 'houses near parks in Manhattan', 'k': 5})
1: final_answer({'answer': 'Here are some of the best houses near parks in Manhattan:\n\n1. **1211 Park Ave**  \n   - Price: $10,250,000  \n   - Beds: 7  \n   - Baths: 6.0  \n   - Size: 5,200 sqft  \n   - Brokered by: Nest Seekers International  \n\n2. **1070 Park Avenue Ave Units MAIS & 1A**  \n   - Price: $750,000  \n   - Beds: 3  \n   - Baths: 2.0  \n   - Size: 1,600 sqft  \n   - Brokered by: Brown Harris Stevens  \n\n3. **591 Park Ave**  \n   - Price: $10,850,000  \n   - Beds: 4  \n   - Baths: 2.37  \n   - Size: 13,000 sqft  \n   - Brokered by: COMPASS  \n\n4. **740 Park Ave # 4 & 5B**  \n   - Price: $48,000,000  \n   - Beds: 5  \n   - Baths: 2.37  \n   - Size: 2,184 sqft  \n   - Brokered by: Corcoran East Side  \n\n5. **80 Park Avenue Ave Unit 10B**  \n   - Price: $699,900  \n   - Beds: 3  \n   - Baths: 1.0  \n   - Size: 558 sqft  \n   - Brokered by: Brown Harris Stevens  \n\nThese properties are located near various

'{"answer": "Here are some of the best houses near parks in Manhattan:\\n\\n1. **1211 Park Ave**  \\n   - Price: $10,250,000  \\n   - Beds: 7  \\n   - Baths: 6.0  \\n   - Size: 5,200 sqft  \\n   - Brokered by: Nest Seekers International  \\n\\n2. **1070 Park Avenue Ave Units MAIS & 1A**  \\n   - Price: $750,000  \\n   - Beds: 3  \\n   - Baths: 2.0  \\n   - Size: 1,600 sqft  \\n   - Brokered by: Brown Harris Stevens  \\n\\n3. **591 Park Ave**  \\n   - Price: $10,850,000  \\n   - Beds: 4  \\n   - Baths: 2.37  \\n   - Size: 13,000 sqft  \\n   - Brokered by: COMPASS  \\n\\n4. **740 Park Ave # 4 & 5B**  \\n   - Price: $48,000,000  \\n   - Beds: 5  \\n   - Baths: 2.37  \\n   - Size: 2,184 sqft  \\n   - Brokered by: Corcoran East Side  \\n\\n5. **80 Park Avenue Ave Unit 10B**  \\n   - Price: $699,900  \\n   - Beds: 3  \\n   - Baths: 1.0  \\n   - Size: 558 sqft  \\n   - Brokered by: Brown Harris Stevens  \\n\\nThese properties are located near various parks, providing great access to green spa